In [1]:
from google.colab import drive
drive.mount('/content/mydrive')

Mounted at /content/mydrive


In [2]:
import os
repo_path = '/content/optimized-summarization'
if not os.path.exists(repo_path):
    !git clone https://github.com/srinisvas/optimized-summarization.git
else:
    print("Repo already exists, skipping clone.")

# Check files inside
os.listdir(repo_path)

Cloning into 'optimized-summarization'...
remote: Enumerating objects: 1625, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 1625 (delta 35), reused 13 (delta 13), pack-reused 1575 (from 2)
Receiving objects: 100% (1625/1625), 108.00 MiB | 21.60 MiB/s, done.
Resolving deltas: 100% (463/463), done.


['optimized-summarization', '.git', '.idea', 'README.md']

In [5]:
!pip install -U transformers accelerate bitsandbytes

import os
import json
import time
import torch
import requests
from typing import Dict, Any, Optional
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- MODEL CONFIGURATION ---
MODEL_NAME = "HuggingFaceTB/SmolLM3-3B"

# Initialize model and tokenizer (using 4-bit quantization to fit on most Colab GPUs)
try:
    print(f"Loading Model: {MODEL_NAME} with 4-bit quantization...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Use BitsAndBytesConfig for 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
    )
    print("Model loaded successfully. Ready for inference.")
except Exception as e:
    print(f"Could not load the model. Ensure you have a GPU runtime and all libraries are installed.")
    print(f"Error details: {e}")
    # Exit or raise error if model loading fails
    raise

# Set max retries for the generation loop, mostly for edge cases
MAX_RETRIES = 3



# 1. Directory containing the original, normalized papers
NORMALIZED_DIR = '/content/optimized-summarization/optimized-summarization/Normalized-papers'

# 2. Output folder for the final summaries
OUTPUT_DIR = '/content/mydrive/MyDrive/NLP Project/ablation_adaptive prompting_differentprompts/'

os.makedirs(OUTPUT_DIR, exist_ok=True)


# --- LLM Generation Function ---

def call_local_llm(chat_messages: list, max_retries: int = MAX_RETRIES) -> Optional[str]:

    #Generates text locally using the loaded  model .

    # 1. template to turn the list of messages into a single prompt string
    prompt = tokenizer.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

    for attempt in range(max_retries):
        try:
            # 2. Tokenize and generate
            input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

            # Start the generation process
            outputs = model.generate(
                **input_ids,
                max_new_tokens=512,
                temperature=0.3,
                do_sample=True, # Use sampling based on temperature
                pad_token_id=tokenizer.eos_token_id
            )

            # 3. Decode the output and clean the text
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # outputs the full prompt + response. We must strip the prompt.
            clean_text = generated_text.replace(prompt, "").strip()

            if "Executive Summary" in clean_text:
                # Ensure we start exactly from the summary title
                clean_text = clean_text.split("Executive Summary", 1)[-1].strip()
                return "Executive Summary\n\n" + clean_text

            # If the specific start phrase isn't found, return the cleaned output anyway
            return clean_text.strip()

        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"   [Attempt {attempt + 1}] Local generation failed ({type(e).__name__}). Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"   [Attempt {attempt + 1}] Local generation failed. Max retries reached. Skipping document.")
                return None
    return None

def load_paper_data(file_name: str) -> Optional[Dict[str, Any]]:
    """
    Ablation Study Modification: Only loads data from the Normalized directory.
    Aggregates all sections into a single 'full_text' field.
    """
    base_id = file_name.replace('.json', '')
    normalized_path = os.path.join(NORMALIZED_DIR, file_name)

    try:
        with open(normalized_path, 'r', encoding='utf-8') as f:
            paper_data = json.load(f)

        # Aggregate every section into one continuous text block
        all_sections = []
        for section in paper_data.get('sections', []):
            title = section.get('title', 'Untitled Section').strip()
            content = section.get('content', '').strip()
            if content:
                all_sections.append(f"### {title}\n{content}")

        full_text = "\n\n".join(all_sections)

        if not full_text:
            print(f"   [Warning] {file_name} appears to be empty. Skipping.")
            return None

        return {
            "paper_id": base_id,
            "full_text": full_text
        }

    except FileNotFoundError:
        print(f"   [Error] File not found: {normalized_path}")
        return None
    except Exception as e:
        print(f"   [Error] Skipping {file_name} due to unexpected error: {e}")
        return None

# --- Prompt Engineering ---

ONE_SHOT_EXAMPLE = """
**[Example] Title: The Role of Quantum Computing in Differential Privacy in IoT Networks**

This paper talks about problems in sharing data while keeping it private in IoT networks. The authors say that normal privacy methods don’t work well when devices share lots of data quickly. They introduce a new system called quantum-resistant differential privacy (QRDP) that uses special encryption and noise to protect data while still allowing analysis. They tested it on simulated traffic data and found it works faster than old methods. The main point is that QRDP balances security and usefulness. Future work will look at hardware tests and checking against quantum attacks.
"""

def create_adaptive_prompt(data: Dict[str, Any]) -> list:
    """
    Constructs the final prompt using the full paper content as context, using a student-style persona.
    """

    # 1. System Instruction (Student Persona)
    system_instruction = (
        "You are a college student reading a research paper. "
        "Your task is to explain the paper in simple words, like a short summary for your class notes. "
        "Be clear and concise, but don’t worry too much about making it very formal or perfectly structured."
    )

    # 2. Student-style Instructions
    student_instructions = (
        "\n\n**Instructions:**\n"
        "1. Read through the paper and understand the main points.\n"
        "2. Write a summary in your own words as if explaining it to a classmate.\n"
        "3. Include what the paper is about, what they did, and what they found.\n"
        "4. Don’t add extra stuff that isn’t in the paper.\n"
        "5. Keep it short and simple; a few paragraphs are enough."
    )

    # 3. Combine instructions and context for the user prompt
    context_injection = f"""
**Target Paper: {data['paper_id']}**
{student_instructions}

--- ONE-SHOT EXAMPLE (Use as a reference for tone and style) ---
{ONE_SHOT_EXAMPLE}
--- END OF EXAMPLE ---

**Full Paper Content:**
{data['full_text']}

---
**TASK:** Write a simple student-style summary based on the paper above. Start your response with the title: **Executive Summary**.
"""

    # 4. Chat Messages List
    chat_messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": context_injection}
    ]
    return chat_messages


# --- Main Orchestration Loop ---

def run_orchestration():
    """
    Iterates through all papers, loads data, prompts LLM, and saves results.
    """

    print(f"Starting Adaptive Summarization with CoT using Local Model: {MODEL_NAME}.")

    processed_count = 0

    try:
        file_list = sorted(os.listdir(NORMALIZED_DIR))
    except FileNotFoundError:
        print(f"\n[FATAL ERROR] Cannot find the main directory: {NORMALIZED_DIR}. Please check the path.")
        return

    for file_name in file_list:
        if file_name.endswith(".json"):
            paper_id = file_name.replace('.json', '')
            print(f"\n--- Orchestrating Summary for {paper_id} ---")

            data = load_paper_data(file_name)

            if data is None:
                continue

            # This creates the list of messages for the chat template
            chat_messages = create_adaptive_prompt(data)

            print("   Generating summary locally on GPU...")
            summary_text = call_local_llm(chat_messages)

            if summary_text:
                output_file_name = f"{paper_id}.txt"
                output_path = os.path.join(OUTPUT_DIR, output_file_name)

                try:
                    with open(output_path, 'w', encoding='utf-8') as f:
                        f.write(summary_text)
                    print(f"   SUCCESS! Summary saved to: {output_path}")
                    processed_count += 1
                except Exception as e:
                    print(f"   [Error] Failed to save output file: {e}")
            else:
                print(f"   [Error] Failed to generate summary for {paper_id}.")

    print(f"\n--- Pipeline execution finished for all documents. Generated {processed_count} final summaries. ---")
    print(f"Check your '{OUTPUT_DIR}' folder in Google Drive!")

if __name__ == "__main__":
    run_orchestration()







Loading Model: HuggingFaceTB/SmolLM3-3B with 4-bit quantization...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully. Ready for inference.
Starting Adaptive Summarization with CoT using Local Model: HuggingFaceTB/SmolLM3-3B.

--- Orchestrating Summary for A Bibliometric View of AI Ethics Development ---
   Generating summary locally on GPU...
   SUCCESS! Summary saved to: /content/mydrive/MyDrive/NLP Project/ablation_adaptive prompting_differentprompts/A Bibliometric View of AI Ethics Development.txt

--- Orchestrating Summary for A Model for Using Ethical Theory to Specify Epistemic Goals for Explainable AI ---
   Generating summary locally on GPU...
   [Attempt 1] Local generation failed (OutOfMemoryError). Retrying in 1s...
   [Attempt 2] Local generation failed (OutOfMemoryError). Retrying in 2s...
   [Attempt 3] Local generation failed. Max retries reached. Skipping document.
   [Error] Failed to generate summary for A Model for Using Ethical Theory to Specify Epistemic Goals for Explainable AI.

--- Orchestrating Summary for A Privacy Impact Assessment Tool for Cloud C

KeyboardInterrupt: 